# Fine-tune Qwen3 trên Google Colab (GPU miễn phí)

**Mục đích:** train LoRA cho Qwen3 trên file `finetune_dataset.jsonl` (do `07-prepare-finetune.js` tạo),
rồi xuất ra file GGUF để chạy trong Ollama và chấm điểm bằng `04-evaluation.js`.

**Thuật ngữ:**

| Từ | Nghĩa |
|---|---|
| **LoRA** | Giữ nguyên model gốc, chỉ train thêm vài ma trận nhỏ (adapter) → nhẹ, nhanh |
| **QLoRA** | LoRA trên model đã nén xuống 4-bit → model 8B vừa GPU T4 16GB của Colab |
| **Unsloth** | Thư viện fine-tune nhanh, ít tốn bộ nhớ GPU, xuất thẳng ra GGUF |
| **GGUF** | Định dạng file model mà Ollama đọc được |
| **Loss** | Độ sai khi model đoán câu trả lời mẫu. Giảm dần = đang học; tụt sát 0 với data ít = có thể đang học thuộc lòng |

**Cách chạy:**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Chạy lần lượt từng ô (Shift + Enter). Ô upload sẽ hỏi file `finetune_dataset.jsonl`.
3. Ô xuất GGUF lưu file vào Google Drive → tải về máy.

In [ ]:
# Cài Unsloth (kéo theo PyTorch, TRL, PEFT, datasets) — mất vài phút
%pip install unsloth

## 1. Cấu hình

In [ ]:
MODEL_NAME = "unsloth/Qwen3-8B"  # Cùng cỡ với qwen3:latest trong Ollama -> so sánh công bằng. Chạy thử nhanh: "unsloth/Qwen3-1.7B"
MAX_SEQ_LENGTH = 1024  # Số token tối đa của 1 mẫu; prompt + SQL ở đây khoảng 350 token
EPOCHS = 3  # Số lần model đọc hết data
LEARNING_RATE = 2e-4  # Bước sửa trọng số: lớn quá dễ "học lố", nhỏ quá học chậm
LORA_RANK = 8  # Kích thước adapter: lớn hơn = học được nhiều hơn nhưng dễ học thuộc lòng hơn
OUTPUT_NAME = "qwen3-nba-sql"

## 2. Upload data

In [ ]:
import json

from google.colab import files

uploaded = files.upload()  # Chọn file finetune_dataset.jsonl
data_file = next(iter(uploaded))
rows = [json.loads(line) for line in open(data_file, encoding="utf-8") if line.strip()]
print(f"Đã đọc {len(rows)} mẫu từ {data_file}")

## 3. Ghép prompt giống `04-evaluation.js`

Mỗi mẫu được ghép thành 1 chuỗi: **prompt ChatML + câu trả lời + `<|im_end|>`**.
Prompt ghép giống hệt `makeQwenPrompt()` trong `04-evaluation.js` (gửi raw, có khối `<think>` rỗng = tắt chế độ suy nghĩ).
Train một kiểu, dùng một kiểu thì model không nhận ra "tình huống quen" → fine-tune mất tác dụng.

In [ ]:
from datasets import Dataset


def make_qwen_prompt(user, system=""):
    """Ghép prompt ChatML của Qwen3 — giống hệt makeQwenPrompt() trong 04-evaluation.js"""
    system_block = f"<|im_start|>system\n{system}<|im_end|>\n" if system else ""
    return (
        system_block
        + f"<|im_start|>user\n{user}<|im_end|>\n"
        + "<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )


def to_text(row):
    """1 mẫu { messages } -> 1 chuỗi train. Thêm <|im_end|> để model học luôn chỗ dừng"""
    messages = {m["role"]: m["content"] for m in row["messages"]}
    return make_qwen_prompt(messages["user"], messages["system"]) + messages["assistant"] + "<|im_end|>"


dataset = Dataset.from_list([{"text": to_text(row)} for row in rows])
SYSTEM_PROMPT = {m["role"]: m["content"] for m in rows[0]["messages"]}["system"]
print(dataset[0]["text"])

## 4. Tải model + gắn LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,  # QLoRA: nén model 4-bit cho vừa GPU T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,  # Hệ số khuếch đại adapter, thường đặt gấp đôi rank
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",  # Tính lại thay vì lưu kết quả trung gian -> đỡ tốn bộ nhớ GPU
    random_state=42,
)

## 5. Trước fine-tune: hỏi thử 1 câu trong bộ đề

Adapter vừa gắn toàn số 0 → model lúc này trả lời y như model gốc.
Câu hỏi lấy từ bộ đề `gold-test-set.jsonl` (không có trong data train) → so trước/sau mới công bằng.

In [ ]:
TEST_QUESTION = "Who is the highest paid NBA player?"  # salary là chữ "$51,915,615" -> phải bóc số mới sắp xếp đúng


def ask(question):
    """Cho model trả lời 1 câu hỏi (greedy = luôn chọn token khả năng cao nhất -> kết quả ổn định)"""
    FastLanguageModel.for_inference(model)
    prompt = make_qwen_prompt(question, SYSTEM_PROMPT)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    output = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


print(ask(TEST_QUESTION))

## 6. Train

Chỉ chấm loss phần **trả lời**: token của system + câu hỏi bị bỏ qua (`train_on_responses_only`)
→ model không phí công học thuộc prompt, chỉ học cách viết SQL.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Cộng dồn 4 bước rồi mới sửa trọng số = như batch 8 mà không tốn thêm bộ nhớ
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=5,  # Vài bước đầu tăng dần learning rate -> đỡ "giật" lúc mới train
        logging_steps=1,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(f"Train xong sau {stats.metrics['train_runtime'] / 60:.1f} phút")

## 7. Sau fine-tune: cùng câu hỏi đó

In [ ]:
print(ask(TEST_QUESTION))

## 8. Xuất GGUF và lưu vào Google Drive

`q4_k_m` = nén 4-bit, cùng kiểu nén với `qwen3:latest` trong Ollama (model 8B khoảng 5GB).
Mất khoảng 30–40 phút với model 8B: tải lại model gốc 16-bit (khoảng 16GB) để gộp LoRA → convert → nén bằng CPU.
Lưu vào Drive vì tải thẳng file vài GB từ Colab hay bị đứt giữa chừng. Cần Drive còn trống khoảng 6GB.
Đợi in dòng "Đã đẩy xong lên Drive" rồi mới tải về; thấy file trên Drive sớm hơn là file chưa đủ.

In [ ]:
import glob
import os
import shutil

from google.colab import drive

model.save_pretrained_gguf(OUTPUT_NAME, tokenizer, quantization_method="q4_k_m")
# Chỉ lấy file đã nén Q4_K_M; file F16 trung gian khoảng 16GB, vượt dung lượng Drive miễn phí
gguf_files = glob.glob("/content/**/*Q4_K_M.gguf", recursive=True)  # Tìm trước khi mount Drive để không quét cả Drive

drive.mount("/content/drive")
for path in gguf_files:
    shutil.copy(path, "/content/drive/MyDrive/")
    print(f"Đã chép: {os.path.basename(path)} ({os.path.getsize(path) / 1e9:.1f}GB)")

# copy xong chưa có nghĩa là đã lên Drive: dữ liệu vẫn đang đẩy ngầm -> chờ đẩy hết rồi mới ngắt
drive.flush_and_unmount()
print("Đã đẩy xong lên Drive, tải về được rồi")

## 9. Chạy trên máy bằng Ollama

1. Tải file `.gguf` từ Google Drive về máy, VD `D:\models\`.
2. Tạo file `Modelfile` (không đuôi) cùng thư mục:
   ```
   FROM ./<tên-file>.gguf
   PARAMETER stop <|im_end|>
   ```
   Không cần khai báo template: `04-evaluation.js` gửi prompt raw (`raw: true`).
3. Nạp vào Ollama: `ollama create qwen3-nba-sql -f Modelfile`
4. Trong `04-evaluation.js` đổi `MODEL` thành `"qwen3-nba-sql"` rồi chạy chấm điểm.